# Kaggle Setup and Usage Guide

## 1. Running Kaggle Docker Locally 

The built Docker images for the Kaggle Python environment (from [kaggle/docker-python](https://github.com/Kaggle/docker-python)),which comes with hundreds of pre-installed packages, are **hosted on Google Container Registry (GCR)**, not Docker Hub or ghcr.io.

**Official image locations:**
- **CPU-only:** `gcr.io/kaggle-images/python`
- **GPU:** `gcr.io/kaggle-gpu-images/python`



**Important notes:**
- Legacy image on Docker Hub ([hub.docker.com/r/kaggle/python](https://hub.docker.com/r/kaggle/python)) is **stale/outdated**. Use the gcr.io images above.
- Tags: Check [releases/tags](https://github.com/Kaggle/docker-python/tags) on GitHub for version-specific tags.


pre-installed packages:

- **Deep Learning:** `torch`, `tensorflow`, `keras`, `transformers`, `timm`
- **Data Science:** `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`
- **Computer Vision:** `opencv-python`, `Pillow`, `albumentations`
- **NLP:** `nltk`, `spacy`, `gensim`
- **Utilities:** `tqdm`, `requests`, `beautifulsoup4`, `plotly`

**Full list:** [kaggle_requirements.txt](https://github.com/Kaggle/docker-python/blob/main/kaggle_requirements.txt)



---

**Useful commands:**

| Action | Command |
|--------|----------|
| View logs | `docker logs --tail 50 temp-kaggle-notebook` |
| Stop | `docker stop temp-kaggle-notebook` |
| Remove container | `docker rm temp-kaggle-notebook` |
| Remove image (~45GB) | `docker rmi gcr.io/kaggle-gpu-images/python:latest` |

**Cleanup:**
```bash
docker container prune -f   # Clean stopped containers
docker image prune -f      # Clean dangling images
```


### Running Kaggle Notebooks Locally

**Pull and run:**

```bash
docker pull gcr.io/kaggle-gpu-images/python
docker stop my-kaggle
```


```bash

# Run with bash first, then manually start jupyter
docker run --rm \
    --gpus all \
    --name my-kaggle \
    -p 8888:8888 \
    -v /home/$USER/anaconda3/envs/PyTorchTutorial/src/projects:/work \
    -w /work \
    -it \
    gcr.io/kaggle-gpu-images/python:latest \
    /bin/bash
```

Then inside the container, run:
```bash
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root --ServerApp.token='kaggle123'
```


 -it gcr.io/kaggle-gpu-images/python:latest 


After starting, open `http://localhost:8888/lab?token=kaggle123`




**Example:**
```python
# In Kaggle notebooks, these are already available:
import torch
# !pip install -q some-new-package
```

---

### Alternate: Build your Customized Kaggle-Compatible Docker Image

**1. Base image:** Built on Kaggle's official GPU image:
```dockerfile
FROM gcr.io/kaggle-gpu-images/python
```

**2. Build locally:**
```bash
cd /home/$USERNAME/anaconda3/envs/PyTorchTutorial/src
docker build -t kaggle-universal:latest .
```

**3. Push to GitHub Container Registry:**
```bash
export GITHUB_USERNAME=behnamasadi
echo $GITHUB_TOKEN | docker login ghcr.io -u $GITHUB_USERNAME --password-stdin
docker tag kaggle-universal:latest ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest
docker push ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest
```

**4. Push to Docker Hub (alternative):**
```bash
docker login
docker tag kaggle-universal:latest <dockerhub-username>/kaggle-universal:latest
docker push <dockerhub-username>/kaggle-universal:latest
```

**5. Use on RunPod:** Configure pod with image `ghcr.io/<github-username>/kaggle-universal:latest` and mount projects to `/workspace/host`.

This setup gives you a single, reusable image that matches Kaggle's stack and works across all Kaggle challenges.

## 2. Kaggle Hardware and GPU

### Available GPUs

| GPU | VRAM | Compute Capability | Best For |
|-----|------|-------------------|----------|
| **NVIDIA Tesla P100** | 16 GB | 6.0 | General deep learning, FP32 |
| **NVIDIA Tesla T4** | 16 GB | 7.5 | Mixed precision (FP16), inference |

You cannot choose which GPU you get. Kaggle assigns P100 or T4 randomly. Both have 16GB VRAM.

**Check which GPU you got:**
```python
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_props.total_memory / 1e9:.1f} GB")
    if "P100" in gpu_name:
        print("Good for FP32 training")
    elif "T4" in gpu_name:
        print("Excellent for FP16/mixed precision (3-4x faster)")
else:
    print("No GPU available")
```

### GPU Comparison

| Aspect | P100 | T4 |
|--------|------|-----|
| **FP32 performance** | 9.3 TFLOPS (better) | 8.1 TFLOPS |
| **FP16 performance** | 18.7 TFLOPS | 65 TFLOPS (3-4x faster) |
| **Memory bandwidth** | 732 GB/s (better) | 320 GB/s |
| **Power** | 250W | 70W (more efficient) |

**Bottom line:** Use mixed precision (FP16); T4 will be much faster, P100 still works.

### System Resources

| Resource | Limit |
|----------|-------|
| **GPU Time** | ~30 hours/week |
| **CPU Cores** | 4 cores |
| **RAM** | 16-30 GB |
| **Disk Space** | ~73 GB (temporary) |
| **Session Time** | 9-12 hours max |
| **Internet** | Enabled (disabled in competitions) |

### Critical Limitations

1. 16 GB VRAM (most common bottleneck)
2. Session timeout (9-12 hours)
3. Disk space (~73 GB)

### GPU Optimization Best Practices

1. Use Mixed Precision Training (FP16)
2. Gradient Accumulation
3. Monitor GPU Memory
4. Find Optimal Batch Size Dynamically
5. Clear Unused Variables and Cache
6. Gradient Checkpointing
7. Use Efficient Data Loading
8. Reduce Model Precision Strategically
9. GPU-Agnostic Code Pattern

## 3. Kaggle Authentication

You need Kaggle API credentials to download datasets.

### Option 1: Local Development (kaggle.json)

1. Go to [Kaggle Account Settings](https://www.kaggle.com/settings/account)
2. Scroll to "API" section
3. Click "Create New Token"
4. Download `kaggle.json`
5. Place it in the correct location:

**Linux/macOS:**
```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
```

**Windows:**
```powershell
# Place kaggle.json in:
C:\Users\<YourUsername>\.kaggle\kaggle.json
```

### Option 2: RunPod/Docker (Environment Variables)

```bash
export KAGGLE_USERNAME="your_username"
export KAGGLE_KEY="your_api_key"
```

Or in Dockerfile:
```dockerfile
ENV KAGGLE_USERNAME=your_username
ENV KAGGLE_KEY=your_api_key
```

### kaggle.json Locations Reference

| Platform | Location |
|----------|----------|
| **Linux** | `~/.kaggle/kaggle.json` or `~/.config/kaggle/kaggle.json` |
| **macOS** | `~/.kaggle/kaggle.json` |
| **Windows** | `C:\Users\<YourUsername>\.kaggle\kaggle.json` |
| **WSL** | `/home/<username>/.kaggle/kaggle.json` |

## Download Data in Notebook


```python
import kagglehub
import os
from pathlib import Path
import numpy as np
import warnings

def download_lungs_dataset():
    """
    Download dataset and find train/val/test directories.

    """
    # Download dataset
    dataset_path = kagglehub.dataset_download("omkarmanohardalvi/lungs-disease-dataset-4-types")
    print(f"✅ Dataset downloaded to: {dataset_path}")
    
    dataset_path = Path(dataset_path)
    
    # Look for train/val/test directories
    possible_train_dirs = list(dataset_path.rglob("train"))
    possible_val_dirs = list(dataset_path.rglob("val"))
    possible_test_dirs = list(dataset_path.rglob("test"))
    
    train_path = possible_train_dirs[0] if possible_train_dirs else None
    val_path = possible_val_dirs[0] if possible_val_dirs else None
    test_path = possible_test_dirs[0] if possible_test_dirs else None
    
    if train_path and val_path:
        print(f"✅ Found train: {train_path}")
        print(f"✅ Found val: {val_path}")
        if test_path:
            print(f"✅ Found test: {test_path}")
        return str(train_path), str(val_path), str(test_path) if test_path else None
    else:
        raise RuntimeError(f"Could not find train/val directories in {dataset_path}")
```

## 4. Kaggle CLI

For local development or custom environments, install the Kaggle API and kagglehub:

```bash
pip install kaggle kagglehub pyyaml
```

- **kaggle**: Official Kaggle API for datasets, competitions, notebooks
- **kagglehub**: Modern dataset downloader (recommended over legacy kaggle CLI for downloads)
- **pyyaml**: For config files

---


## 5. Configuration

Edit `config/train.yaml`:

```yaml
data:
  kaggle_dataset: "owner/dataset-name"  # e.g., masoudnickparvar/brain-tumor-mri-dataset
  path: "./data/train"                   # Relative to kaggle_structure directory
```

## 6. Finding Kaggle Datasets

### Method 1: Kaggle Website

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets)
2. Search with keywords:
   - **Medical:** `brain`, `tumor`, `MRI`, `liver`, `kidneys`, `heart`, `lungs`, `vessel`, `lesion`, `COVID`, `lung`, `opacities`, `breast`, `cancer`, `Ultrasound`, `histopathology`, `segmentation`, `medical imaging`, `retinal`, `OCT`, `tomography`, `endoscopy`
   - **Computer Vision:** `object detection`, `segmentation`, `saliency`, `image similarity`
   - **3D Vision:** `LiDAR`, `SLAM`, `Photogrammetry`, `stereo`, `monocular depth`
3. Dataset name is in the URL: `kaggle.com/datasets/OWNER/DATASET-NAME`
4. Use `OWNER/DATASET-NAME` in `config/train.yaml`

### Method 2: Kaggle CLI

**Search:**
```bash
kaggle datasets list -s "MRI"
kaggle datasets list -s "MRI" --sort-by votes
kaggle datasets list -s "medical imaging" --sort-by votes -p 3
kaggle datasets list -s "segmentation" --max-size 50
```

Sort options: `hottest`, `votes`, `updated`, `active`, `published`

**Download:**
```bash
kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
```

### Method 3: Browse by Category

- [Sports](https://www.kaggle.com/datasets?topic=sports)
- [Medicine](https://www.kaggle.com/datasets?topic=health)
- [Computer Vision](https://www.kaggle.com/datasets?topic=computer-vision)

## 6. Kaggle Dataset Downloader (kagglehub)

This script creates symlinks from your project data directory to the Kaggle cache.

**What it does:**
1. Reads `config/train.yaml`
2. Downloads dataset via kagglehub (cache: `~/.cache/kagglehub/`)
3. Creates symlink from `kaggle_structure/data/train` to the downloaded dataset
4. Works locally (kaggle.json) and on RunPod (KAGGLE_USERNAME, KAGGLE_KEY)

[Downloader Script](scripts/kaggle_dataset_downloader.py)

## 7. Uploading and Creating Your Own Dataset

1. Go to [Kaggle Datasets](https://www.kaggle.com/datasets) and click **New Dataset**
2. Upload files or connect to cloud storage
3. Add title, description, and metadata
4. Use the [Kaggle API](https://www.kaggle.com/docs/api) for programmatic uploads

**References:**
- [Kaggle API documentation](https://www.kaggle.com/docs/api#installation)
- [RVP Group SLAM dataset](https://rvp-group.net/slam-dataset.html) (example)